In [1]:
import SimpleITK as sitk
import matplotlib.pyplot as plt
import nibabel as nib
from pathlib import Path
import numpy as np
import yaml
import time

In [2]:
from totalsegmentator.python_api import totalsegmentator

In [3]:
def get_organ_task (organ_yaml: str):
    with open(organ_yaml,"r") as f:
        organs = yaml.safe_load(f)
        
    organs_in_total = []
    organs_in_head = []
    tasks = set()
    for organ in organs["organs"]:
        tasks.add(organ["task"])
        if (organ["task"]) == "total":
            organs_in_total.append(organ["source_class"])
        else:
            organs_in_head.append(organ["source_class"])
    return organs_in_total, organs_in_head, tasks



In [7]:
def get_params (data_path: str, organ_yaml: str):
    data_folder = Path(data_path)
    input_folder = data_folder / "input"
    output_folder = data_folder / "output" 
    params = []
    organ_task = get_organ_task(organ_yaml)
    tasks = organ_task[2]
    roi_subset = organ_task[0]
    for subject_dir in sorted(input_folder.iterdir()): # Subject = quadra 1-48
        if subject_dir.is_dir():
            # print(subject_dir)
            subject_img_dirs = sorted(subject_dir.glob("*.nii.gz"))
            if not subject_img_dirs:
                print ("WARNING!!! images not found in this dir:", subject_dir) 
            else:
                for subject_img_dir in subject_img_dirs:
                    session_name = subject_img_dir.name.removesuffix("_CT-AC.nii.gz") # Session = test, retest
                    subject_mask_dir = output_folder / subject_dir.name / session_name
                    subject_mask_dir.mkdir(parents=True, exist_ok=True)
                    for task in tasks:
                        param = {
                            "subject_id": subject_dir.name,
                            "session": session_name,
                            "task": task,
                            "input": subject_img_dir,
                            "output": subject_mask_dir,
                            "device": "gpu"
                        }
                        if task == "total":
                            param["roi_subset"] = roi_subset
                        params.append(param)
    
    
    return params

parameters = get_params (data_path="data", organ_yaml="organs.yaml")

WARNING!!! images not found in this dir: data/input/example_quadra_22
WARNING!!! images not found in this dir: data/input/example_quadra_23


In [8]:
parameters

[{'subject_id': 'example_quadra_21',
  'session': 'retest',
  'task': 'total',
  'input': PosixPath('data/input/example_quadra_21/retest_CT-AC.nii.gz'),
  'output': PosixPath('data/output/example_quadra_21/retest'),
  'device': 'gpu',
  'roi_subset': ['brain',
   'skull',
   'vertebrae_C1',
   'vertebrae_C4',
   'vertebrae_C7',
   'lung_upper_lobe_left',
   'lung_lower_lobe_left',
   'lung_upper_lobe_right',
   'lung_middle_lobe_right',
   'lung_lower_lobe_right',
   'heart',
   'esophagus',
   'vertebrae_T4',
   'vertebrae_T8',
   'vertebrae_T12',
   'rib_left_6',
   'rib_right_6',
   'liver',
   'kidney_left',
   'kidney_right',
   'pancreas',
   'duodenum',
   'small_bowel',
   'colon',
   'vertebrae_L1',
   'vertebrae_L3',
   'vertebrae_L5',
   'urinary_bladder',
   'prostate',
   'sacrum',
   'hip_left',
   'hip_right']},
 {'subject_id': 'example_quadra_21',
  'session': 'retest',
  'task': 'head_glands_cavities',
  'input': PosixPath('data/input/example_quadra_21/retest_CT-AC.nii

In [9]:
start_time = time.perf_counter()
for parameter in parameters:
    print (
        "input = ", parameter["input"], "\n",
        "output = ", parameter["output"], "\n",
        "task = ", parameter["task"], "\n",
        "device = ", parameter["device"], "\n"
        "roi_subset = ", parameter.get("roi_subset", None)
    )
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Segmentation time: {execution_time:.6f} seconds")

input =  data/input/example_quadra_21/retest_CT-AC.nii.gz 
 output =  data/output/example_quadra_21/retest 
 task =  total 
 device =  gpu 
roi_subset =  ['brain', 'skull', 'vertebrae_C1', 'vertebrae_C4', 'vertebrae_C7', 'lung_upper_lobe_left', 'lung_lower_lobe_left', 'lung_upper_lobe_right', 'lung_middle_lobe_right', 'lung_lower_lobe_right', 'heart', 'esophagus', 'vertebrae_T4', 'vertebrae_T8', 'vertebrae_T12', 'rib_left_6', 'rib_right_6', 'liver', 'kidney_left', 'kidney_right', 'pancreas', 'duodenum', 'small_bowel', 'colon', 'vertebrae_L1', 'vertebrae_L3', 'vertebrae_L5', 'urinary_bladder', 'prostate', 'sacrum', 'hip_left', 'hip_right']
input =  data/input/example_quadra_21/retest_CT-AC.nii.gz 
 output =  data/output/example_quadra_21/retest 
 task =  head_glands_cavities 
 device =  gpu 
roi_subset =  None
input =  data/input/example_quadra_21/test_CT-AC.nii.gz 
 output =  data/output/example_quadra_21/test 
 task =  total 
 device =  gpu 
roi_subset =  ['brain', 'skull', 'vertebrae

In [ ]:
start_time = time.perf_counter()
for parameter in parameters:
    if __name__ == "__main__":
        totalsegmentator(
            input = parameter["input"],
            output = parameter["output"],
            task = parameter["task"],
            roi_subset = parameter.get("roi_subset", None),
            device = parameter["device"],
        )
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Segmentation time: {execution_time:.6f} seconds")


In [ ]:
start_time = time.perf_counter()
for parameter in parameters:
    if __name__ == "__main__":
        totalsegmentator(
            input = parameter["input"],
            output = parameter["output"],
            task = parameter["task"],
            device = parameter["device"],
        )
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Segmentation time: {execution_time:.6f} seconds")


Segmentation time: 463.452297 seconds
Segmentation time: 1521.024349 seconds